In [ ]:
# UPAR27 Track1 PAR LARGE v7 | aaravsubedi | ViT-L/14 + best-mA + strong augHF_TOKEN = os.environ.get("HF_TOKEN", "hf_YOUR_TOKEN_HERE")  # was hardcoded during Kaggle runs; set HF_TOKEN env/secretHF_REPO="ShivRamSaud/upar-track1-par-large"import os, gc, time, jsonos.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

In [ ]:
!pip install -q transformers huggingface_hub timm scikit-learn tqdm gdown
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))

In [ ]:
from huggingface_hub import login, create_repo, snapshot_download, HfApi
login(token=HF_TOKEN)
create_repo(HF_REPO, exist_ok=True, private=False)
print('HF ok', HF_REPO)

In [ ]:
# Annotations (light, 1.7MB) + images via Kaggle datasets if attached, else download
!rm -rf /tmp/upar && git clone --depth 1 https://github.com/speckean/UPAR-Challenge-2027 /tmp/upar
!ls /kaggle/input 2>&1 | head -30
!ls /tmp/upar/data/annotations/task1/train 2>&1 | head
import pathlib
# Resolve image roots: try kaggle inputs first
CANDS=["/kaggle/input/market1501","/kaggle/input/pa100k","/kaggle/input/upar2027-dev/data","/tmp/upar/data"]
for c in CANDS+ ["/kaggle/input"]:
  p=pathlib.Path(c)
  if p.exists(): print(c, "->", [x.name for x in list(p.iterdir())[:10]])

In [ ]:
import torch.nn as nn
from transformers import CLIPModel, CLIPProcessor
MID='openai/clip-vit-large-patch14'
clip=CLIPModel.from_pretrained(MID)
try:
 clip.vision_model.gradient_checkpointing_enable()
except: pass
print('CLIP ok, image_size', clip.config.vision_config.image_size)
class CLIP_PAR(nn.Module):
 def __init__(self,clip,n=40):
  super().__init__(); self.clip=clip; d=clip.config.projection_dim
  self.head=nn.Sequential(nn.Dropout(0.3),nn.Linear(d,512),nn.GELU(),nn.Dropout(0.2),nn.Linear(512,n))
 def forward(self,pix):
  v=self.clip.get_image_features(pixel_values=pix)
  import torch as _t
  _o=v
  v=_o[0] if isinstance(_o,(tuple,list)) else (_o.pooler_output if hasattr(_o,'pooler_output') and _o.pooler_output is not None else (_o.last_hidden_state[:,0] if hasattr(_o,'last_hidden_state') else (_o.image_embeds if hasattr(_o,'image_embeds') else _o)))
  v=v if _t.is_tensor(v) else _t.as_tensor(v)
  return self.head(v)
model=CLIP_PAR(clip).cuda().train()
print(f'{sum(p.numel() for p in model.parameters())/1e6:.1f}M')

In [ ]:
import csv, pathlib
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
TF=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(0.25,0.25,0.25),transforms.ToTensor(),transforms.Normalize((0.481,0.457,0.408),(0.268,0.261,0.275)),transforms.RandomErasing(p=0.25)])
TFV=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize((0.481,0.457,0.408),(0.268,0.261,0.275))])
ANNOT="/tmp/upar/data/annotations"
IMG_ROOTS=["/kaggle/input/market1501","/kaggle/input/pa100k","/tmp/upar/data","/kaggle/input"]
for r in list(pathlib.Path("/kaggle/input").glob("*")) if pathlib.Path("/kaggle/input").exists() else []:
 if r.is_dir() and str(r) not in IMG_ROOTS: IMG_ROOTS.append(str(r))
print(IMG_ROOTS)
import pathlib as _pl
INDEX={}
for _base in IMG_ROOTS:
 try:
  for _f in _pl.Path(_base).rglob("*.jpg"):
   if _f.name not in INDEX: INDEX[_f.name]=str(_f)
 except Exception: pass
print("indexed", len(INDEX), "images")
def find_img(rel):
 rel=rel.lstrip("# ").strip()
 for b in ["/tmp/upar/data"]+IMG_ROOTS:
  c=_pl.Path(b)/rel
  if c.exists(): return c
 bn=_pl.Path(rel).name
 if bn in INDEX: return _pl.Path(INDEX[bn])
 return None
class UPAR(Dataset):
 def __init__(self,split,tf):
  rows=list(csv.reader(open(pathlib.Path(ANNOT)/f"task1/{split}/gt.csv")))
  self.rows=[r for r in rows if find_img(r[0]) is not None]
  print(split, f"kept {len(self.rows)}/{len(rows)}")
  self.tf=tf
 def __len__(self): return len(self.rows)
 def __getitem__(self,i):
  r=self.rows[i]; y=__import__("torch").tensor([float(x) for x in r[1:41]],dtype=__import__("torch").float32)
  return self.tf(Image.open(find_img(r[0])).convert("RGB")), y
tr=UPAR("train",TF); va=UPAR("val",TFV)
print("train/val",len(tr),len(va))


In [ ]:
from torch.utils.data import DataLoader
from huggingface_hub import HfApi
import torch.nn.functional as F
def asl(pred,tgt,gneg=2):
 xs=torch.sigmoid(pred); pt=xs*tgt+(1-xs)*(1-tgt); w=(1-pt).pow((1-tgt)*gneg)
 return (F.binary_cross_entropy_with_logits(pred,tgt,reduction='none')*w).mean()
os.makedirs('/tmp/hf_ckpt',exist_ok=True)
# resume
step0=0
try:
 snapshot_download(HF_REPO,local_dir='/tmp/hf_ckpt',allow_patterns=['pytorch_model.bin','step.json'])
 import os as _os
 if _os.path.exists('/tmp/hf_ckpt/pytorch_model.bin'):
  model.load_state_dict(torch.load('/tmp/hf_ckpt/pytorch_model.bin',map_location='cuda')); print('resumed')
 if _os.path.exists('/tmp/hf_ckpt/step.json'): step0=json.load(open('/tmp/hf_ckpt/step.json')).get('step',0)
except Exception as e: print('no resume',e)
loader=DataLoader(tr,batch_size=16,shuffle=True,num_workers=4,pin_memory=True,drop_last=True)
vloader=DataLoader(va,batch_size=64,num_workers=2,pin_memory=True)
opt=torch.optim.AdamW([{'params':model.clip.parameters(),'lr':2e-5},{'params':model.head.parameters(),'lr':8e-5}],weight_decay=0.05)
scaler=torch.amp.GradScaler('cuda')
api=HfApi(); step=step0; t0=time.time()
for ep in range(12):
 best=0
 if time.time()-t0>6.5*3600: print('guard break'); break
 model.train()
 for bi,(x,y) in enumerate(loader):
  x,y=x.cuda(non_blocking=True),y.cuda(non_blocking=True)
  with torch.amp.autocast('cuda'): loss=asl(model(x),y)/4
  scaler.scale(loss).backward()
  if (bi+1)%4==0: scaler.step(opt); scaler.update(); opt.zero_grad(); step+=1
  if step%500==0 and step>step0:
   torch.save(model.state_dict(),'/tmp/hf_ckpt/pytorch_model.bin')
   open('/tmp/hf_ckpt/step.json','w').write(json.dumps({'step':step,'epoch':ep}))
   try: api.upload_folder(repo_id=HF_REPO,folder_path='/tmp/hf_ckpt',commit_message=f'step {step}')
   except Exception as e: print('push err',e)
 model.eval(); cor=tot=0
 with torch.no_grad():
  for x,y in vloader:
   pr=(torch.sigmoid(model(x.cuda())).cpu()>0.5)==(y>0.5); cor+=pr.float().mean().item(); tot+=1
 mA=cor/max(1,tot); print(f'ep {ep} acc {mA:.4f} step {step}')
 if mA>best: best=mA; torch.save(model.state_dict(),'/tmp/hf_ckpt/best_model.bin'); print(f'NEW BEST {best:.4f}')
 torch.save(model.state_dict(),'/tmp/hf_ckpt/pytorch_model.bin')
 open('/tmp/hf_ckpt/step.json','w').write(json.dumps({'step':step,'epoch':ep,'acc':mA}))
 api.upload_folder(repo_id=HF_REPO,folder_path='/tmp/hf_ckpt',commit_message=f'ep {ep} acc {mA:.3f}')
print('DONE HF:',HF_REPO)
import shutil
shutil.copy('/tmp/hf_ckpt/best_model.bin','/tmp/hf_ckpt/pytorch_model.bin')
api.upload_folder(repo_id=HF_REPO,folder_path='/tmp/hf_ckpt',commit_message='final BEST pushed')
print('BEST restored as pytorch_model.bin')

In [ ]:
model.eval()
import csv
with torch.no_grad(), open('/tmp/pred_val.csv','w',newline='') as f:
 w=csv.writer(f)
 for i in range(min(len(va),5000)):
  x,_=va[i]; p=(torch.sigmoid(model(x.unsqueeze(0).cuda())).cpu().numpy()[0]>0.5).astype(int)
  w.writerow([va.rows[i][0]]+p.tolist())
HfApi().upload_file(repo_id=HF_REPO,path_in_repo='pred_val.csv',path_or_fileobj='/tmp/pred_val.csv',commit_message='preds')
print('preds on HF')